In [1]:
!pip install accelerate

In [ ]:
# 1. INSTALLATION & SETUP
print("⏳ Installing dependencies...")
!pip install -q langchain langchain-community faiss-cpu rank_bm25 sentence-transformers bert_score torch numpy pandas tqdm accelerate transformers bitsandbytes

import os
import json
import pickle
import gc
import numpy as np
import torch
import faiss
from tqdm.auto import tqdm
from google.colab import drive
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from bert_score import score
from transformers import AutoTokenizer, AutoModelForCausalLM

# Mount Drive
drive.mount('/content/drive')

# 2. CONFIGURATION & PATHS
BASE_PATH = '/content/drive/My Drive/anlp3/'
if not os.path.exists(BASE_PATH):
    os.makedirs(BASE_PATH)

# File Paths
CORPUS_FILE = os.path.join(BASE_PATH, 'corpus.jsonl')
TEST_FILE = os.path.join(BASE_PATH, 'test.jsonl')
CHUNKED_FILE = 'corpus_chunked.jsonl' 

# Model Names (Optimized for Free Colab)
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'
GEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Settings
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
TOP_K_RETRIEVAL = 10      # For single methods
TOP_K_HYBRID_POOL = 20    # Pool size for hybrid
TOP_K_FINAL = 10          # Final size after reranking

# 3. HELPER FUNCTIONS
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

# 4. STEP 1: PREPROCESSING & CHUNKING
def step_1_chunking():
    print("\n--- STEP 1: Chunking Corpus ---")
    if os.path.exists(CHUNKED_FILE):
        print(f"Found existing chunked file: {CHUNKED_FILE}")
        return load_jsonl(CHUNKED_FILE)

    raw_corpus = load_jsonl(CORPUS_FILE)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunked_data = []
    for doc in tqdm(raw_corpus, desc="Chunking"):
        chunks = splitter.split_text(doc['text'])
        for i, txt in enumerate(chunks):
            chunked_data.append({
                "id": f"{doc['id']}_{i}",
                "doc_id": doc['id'],
                "text": txt
            })

    save_jsonl(chunked_data, CHUNKED_FILE)
    print(f"✅ Chunking complete. Total chunks: {len(chunked_data)}")
    return chunked_data

# 5. STEP 2: INDEXING (BM25 & FAISS)
def step_2_indexing(chunks):
    print("\n--- STEP 2: Indexing ---")

    # --- BM25 ---
    bm25_path = os.path.join(BASE_PATH, 'bm25_index.pkl')
    print("Building/Loading BM25 Index...")
    tokenized_corpus = [c['text'].split() for c in chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(bm25_path, 'wb') as f:
        pickle.dump(bm25, f)

    # --- FAISS ---
    faiss_path = os.path.join(BASE_PATH, 'faiss_index.bin')
    print("Building FAISS Index (Embeddings)...")

    model = SentenceTransformer(EMBEDDING_MODEL, device='cuda')
    texts = [c['text'] for c in chunks]

    # Encode in batches
    embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension) # Inner Product (Cosine similarity if normalized)
    faiss.normalize_L2(embeddings) # Normalize for Cosine Similarity
    index.add(embeddings)

    faiss.write_index(index, faiss_path)

    del model, embeddings
    clean_memory()

    return bm25, index

# 6. STEP 3: RETRIEVAL (BM25, FAISS, HYBRID)
def step_3_retrieval(chunks, test_data, bm25, faiss_index):
    print("\n--- STEP 3: Retrieval & Reranking ---")

    # Load Embedding model again for Query Encoding
    embedder = SentenceTransformer(EMBEDDING_MODEL, device='cuda')

    # Load Reranker
    print("Loading Reranker Model...")
    reranker = CrossEncoder(RERANKER_MODEL, device='cuda', num_labels=1)

    bm25_results = []
    faiss_results = []
    hybrid_results = []

    # Pre-calculate query embeddings for speed
    print("Encoding queries...")
    questions = [t['question'] for t in test_data]
    q_embeddings = embedder.encode(questions, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    faiss.normalize_L2(q_embeddings)

    for i, item in enumerate(tqdm(test_data, desc="Processing Queries")):
        question = item['question']
        gold = item['answers']

        # 1. BM25 Retrieval
        bm25_scores = bm25.get_scores(question.split())
        # Top-K for BM25 Only
        top_bm25_idx = np.argsort(bm25_scores)[::-1][:TOP_K_RETRIEVAL]
        bm25_entry = {
            "question": question, "gold_answers": gold,
            "retrieved_chunks": [{"id": chunks[idx]['id'], "text": chunks[idx]['text']} for idx in top_bm25_idx]
        }
        bm25_results.append(bm25_entry)

        # 2. FAISS Retrieval
        D, I = faiss_index.search(q_embeddings[i].reshape(1, -1), TOP_K_RETRIEVAL)
        faiss_entry = {
            "question": question, "gold_answers": gold,
            "retrieved_chunks": [{"id": chunks[idx]['id'], "text": chunks[idx]['text']} for idx in I[0] if idx != -1]
        }
        faiss_results.append(faiss_entry)

        # 3. Hybrid + Reranking
        # Get Candidates (Top-20 BM25 + Top-20 FAISS)
        top_bm25_pool = np.argsort(bm25_scores)[::-1][:TOP_K_HYBRID_POOL]
        D_pool, I_pool = faiss_index.search(q_embeddings[i].reshape(1, -1), TOP_K_HYBRID_POOL)

        # Merge unique chunks
        unique_candidates = {}

        for idx in top_bm25_pool:
            unique_candidates[chunks[idx]['id']] = chunks[idx]['text']

        for idx in I_pool[0]:
            if idx != -1:
                unique_candidates[chunks[idx]['id']] = chunks[idx]['text']

        # Prepare for Reranker
        candidate_texts = list(unique_candidates.values())
        candidate_ids = list(unique_candidates.keys())
        pairs = [[question, txt] for txt in candidate_texts]

        if not pairs:
            hybrid_results.append({"question": question, "gold_answers": gold, "retrieved_chunks": []})
            continue

        # Predict scores
        rerank_scores = reranker.predict(pairs)

        # Sort and take Top-10
        sorted_indices = np.argsort(rerank_scores)[::-1][:TOP_K_FINAL]
        final_chunks = [{"id": candidate_ids[ix], "text": candidate_texts[ix]} for ix in sorted_indices]

        hybrid_results.append({
            "question": question,
            "gold_answers": gold,
            "retrieved_chunks": final_chunks
        })

    # Cleanup
    del embedder, reranker
    clean_memory()

    return bm25_results, faiss_results, hybrid_results

# 7. STEP 4: GENERATION (LLM)
def generate_answers(dataset, output_path):
    print(f"\n--- STEP 4: Generation with {GEN_MODEL_NAME} ---")

    # Load LLM
    tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    generated_data = []

    for entry in tqdm(dataset, desc="Generating"):
        question = entry['question']
        chunks = entry['retrieved_chunks']

        # Combine context
        context_text = "\n".join([f"- {c['text']}" for c in chunks])

        # System Prompt




        messages = [
            {"role": "system", "content": """You are an intelligent assistant. Answer the users question 
strictly and only based on the provided texts. If the answer 
is not found in the texts, say: 
There is not enough information in the available sources."""},
            {"role": "user", "content": f"contexts:\n{context_text}\n\nquestion: {question}"}
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=150,
                temperature=0.1, # Low temperature for factual consistency
                do_sample=False
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        generated_data.append({
            **entry,
            "generated_answer": response
        })

    save_jsonl(generated_data, output_path)

    # Cleanup LLM to free memory for Evaluation
    del model, tokenizer
    clean_memory()
    print(f"✅ Generation saved to {output_path}")
    return generated_data

# 8. STEP 5: EVALUATION
def evaluate_results(file_path, method_name, judge_model, judge_tokenizer):
    print(f"\nEvaluating: {method_name}")
    data = load_jsonl(file_path)

    cands = [d['generated_answer'] for d in data]
    # Handle multiple gold answers by taking the first one for BERTScore
    refs = [d['gold_answers'][0] if isinstance(d['gold_answers'], list) else d['gold_answers'] for d in data]

    # 1.BERTScore
    P, R, F1 = score(cands, refs, model_type="bert-base-multilingual-cased", verbose=False, device='cuda', batch_size=32)
    
    bs_precision = P.mean().item()  
    bs_recall = R.mean().item()     
    bs_f1 = F1.mean().item()

    # 2. LLM Judge
    judge_scores = []

    for item in tqdm(data, desc="LLM Judge"):
        q = item['question']
        gen = item['generated_answer']
        gold = item['gold_answers'][0] if isinstance(item['gold_answers'], list) else item['gold_answers']

        prompt = f"""You are an evaluator.
Question: {q}
Correct Answer: {gold}
Predicted Answer: {gen}

Is the predicted answer semantically correct based on the correct answer? Answer only '1' for yes or '0' for no."""

        messages = [{"role": "user", "content": prompt}]
        text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = judge_tokenizer([text], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = judge_model.generate(**inputs, max_new_tokens=5, do_sample=False)
            res = judge_tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

        if "1" in res:
            judge_scores.append(1)
        else:
            judge_scores.append(0)

    avg_judge = sum(judge_scores) / len(judge_scores) if judge_scores else 0

    print(f"{method_name} -> P: {bs_precision:.4f} | R: {bs_recall:.4f} | F1: {bs_f1:.4f} | LLM Judge: {avg_judge:.4f}")
    
    return bs_precision, bs_recall, bs_f1, avg_judge

# MAIN EXECUTION FLOW
if __name__ == "__main__":

    # 1. Chunking
    chunks = step_1_chunking()

    # 2. Indexing
    bm25_idx, faiss_idx = step_2_indexing(chunks)

    # 3. Retrieval
    test_data = load_jsonl(TEST_FILE)
    res_bm25, res_faiss, res_hybrid = step_3_retrieval(chunks, test_data, bm25_idx, faiss_idx)

    # 4. Generation
    # We run generation sequentially to save RAM
    # Note: We pass paths because the function reloads the model
    # Saving intermediate retrieval results
    save_jsonl(res_bm25, 'bm25_retrieved.jsonl')
    save_jsonl(res_faiss, 'faiss_retrieved.jsonl')
    save_jsonl(res_hybrid, 'hybrid_retrieved.jsonl')

    gen_bm25 = generate_answers(res_bm25, 'gen_bm25.jsonl')
    gen_faiss = generate_answers(res_faiss, 'gen_faiss.jsonl')
    gen_hybrid = generate_answers(res_hybrid, 'gen_hybrid.jsonl')

    # 5. Evaluation
    print("\n--- STEP 5: Final Evaluation ---")
    
    p_bm25, r_bm25, f1_bm25, judge_bm25 = evaluate_results('gen_bm25.jsonl', "Lexical (BM25)", judge_model, judge_tokenizer)
    p_faiss, r_faiss, f1_faiss, judge_faiss = evaluate_results('gen_faiss.jsonl', "Semantic (FAISS)", judge_model, judge_tokenizer)
    p_hybrid, r_hybrid, f1_hybrid, judge_hybrid = evaluate_results('gen_hybrid.jsonl', "Hybrid + Rerank", judge_model, judge_tokenizer)

    report = f"""
    Evaluation Report
    =================
    Lexical (BM25):
        BERTScore Precision: {p_bm25:.4f}
        BERTScore Recall:    {r_bm25:.4f}
        BERTScore F1:        {f1_bm25:.4f}
        LLM Judge Accuracy:  {judge_bm25:.4f}

    Semantic (FAISS):
        BERTScore Precision: {p_faiss:.4f}
        BERTScore Recall:    {r_faiss:.4f}
        BERTScore F1:        {f1_faiss:.4f}
        LLM Judge Accuracy:  {judge_faiss:.4f}

    Hybrid (Reranked):
        BERTScore Precision: {p_hybrid:.4f}
        BERTScore Recall:    {r_hybrid:.4f}
        BERTScore F1:        {f1_hybrid:.4f}
        LLM Judge Accuracy:  {judge_hybrid:.4f}
    """
    
    print(report)
    with open(os.path.join(BASE_PATH, 'final_report.txt'), 'w') as f:
        f.write(report)

⏳ Installing dependencies...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- STEP 1: Chunking Corpus ---


Chunking:   0%|          | 0/20000 [00:00<?, ?it/s]

✅ Chunking complete. Total chunks: 88308

--- STEP 2: Indexing ---
Building/Loading BM25 Index...
Building FAISS Index (Embeddings)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

Generating:   0%|          | 0/3000 [00:00<?, ?it/s]

✅ Generation saved to gen_faiss.jsonl

--- STEP 4: Generation with Qwen/Qwen2.5-1.5B-Instruct ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating:   0%|          | 0/3000 [00:00<?, ?it/s]

KeyboardInterrupt: 